# Sandy Football — Data Exploration

Explores the `football` schema we just backfilled from API-Football
(senior men's national-team competitions, 2019–2026).

Tables:
- **football.matches** — one row per fixture (goals, competition, weight, status)
- **football.teams** — national teams (name, country, fifa_code)
- **football.match_stats** — per-team match stats (corners/cards/possession) — *trickling in*
- **football.team_ratings** — Dixon-Coles attack/defense snapshots — *populated by `fit_and_persist`*
- **football.match_predictions** — predictions + reconciled outcomes — *populated by the predictor*

Run on the EC2 with the DB env vars set.

## 1. Connect

In [1]:
import pandas as pd
from sqlalchemy import text
from sandy.config import load_config
from sandy.db import create_engine

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
cfg = load_config()
engine = create_engine(cfg)

def q(sql, **params):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

print("Connected to", cfg.database.name)

Connected to sandy


## 2. Overview — row counts per table

In [2]:
q('''
SELECT 'matches' AS table, count(*) AS rows FROM football.matches
UNION ALL SELECT 'teams', count(*) FROM football.teams
UNION ALL SELECT 'match_stats', count(*) FROM football.match_stats
UNION ALL SELECT 'team_ratings', count(*) FROM football.team_ratings
UNION ALL SELECT 'match_predictions', count(*) FROM football.match_predictions
ORDER BY rows DESC
''')

,table,rows
0,matches,3956
1,match_predictions,3055
2,teams,373
3,team_ratings,366
4,match_stats,0


In [4]:
q('''
SELECT * FROM football.match_predictions
''')

,id,fixture_id,match_date,home_team_id,away_team_id,predicted_at_utc,lambda_home,lambda_away,p_home_win,p_draw,p_away_win,p_over_1_5,p_over_2_5,p_over_3_5,p_over_4_5,p_btts,most_likely_home,most_likely_away,scoreline,feature_vector,p_correct,actual_home_goals,actual_away_goals,actual_total_goals,actual_result,actual_btts,was_correct_result,was_correct_over_2_5,was_correct_btts,was_correct_score,outcome_filled_at_utc
0,1665,1138697,2024-01-13,1501,1513,2026-06-15 03:48:51.835115+00:00,1.542104,0.417838,0.652450,0.253035,0.094516,0.585887,0.312482,0.135721,0.049111,0.271312,1,0,"[{'a': 0, 'h': 1, 'p': 0.2144}, {'a': 0, 'h': ...","{'atk_away': -0.3759215795532071, 'atk_home': ...",None,2,0,2,H,False,True,True,True,False,2026-06-15 03:49:40.663774+00:00
1,1670,1138698,2024-01-14,19,1521,2026-06-15 03:48:51.837382+00:00,1.443273,1.492426,0.362530,0.253112,0.384357,0.794623,0.562250,0.338365,0.174050,0.595697,1,1,"[{'a': 1, 'h': 1, 'p': 0.1179}, {'a': 2, 'h': ...","{'atk_away': 0.26970752318310914, 'atk_home': ...",None,1,1,2,D,True,False,False,True,True,2026-06-15 03:49:40.663774+00:00
2,1671,1138699,2024-01-14,32,1512,2026-06-15 03:48:51.837813+00:00,2.535384,0.598416,0.791889,0.143496,0.064615,0.822023,0.606082,0.382673,0.207643,0.416708,2,0,"[{'a': 0, 'h': 2, 'p': 0.14}, {'a': 0, 'h': 3,...","{'atk_away': 0.20112122659788054, 'atk_home': ...",None,2,2,4,D,True,False,True,False,False,2026-06-15 03:49:40.663774+00:00
3,1672,1138700,2024-01-14,1504,1533,2026-06-15 03:48:51.838349+00:00,1.535616,1.254777,0.434631,0.257894,0.307475,0.770988,0.528253,0.305925,0.150830,0.564642,1,1,"[{'a': 1, 'h': 1, 'p': 0.122}, {'a': 1, 'h': 2...","{'atk_away': 0.3611452482538169, 'atk_home': 0...",None,1,2,3,A,True,False,True,True,False,2026-06-15 03:49:40.663774+00:00
4,1676,1138701,2024-01-15,13,1492,2026-06-15 03:48:51.840132+00:00,3.403495,0.566533,0.890268,0.079399,0.030333,0.907266,0.757274,0.560293,0.364789,0.419256,3,0,"[{'a': 0, 'h': 3, 'p': 0.1241}, {'a': 0, 'h': ...","{'atk_away': -0.17482894175415853, 'atk_home':...",None,3,0,3,H,False,True,True,True,True,2026-06-15 03:49:40.663774+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3050,307,901637,2022-07-17,1531,1520,2026-06-15 03:48:19.693099+00:00,0.581768,0.215125,0.373761,0.513862,0.112377,0.192559,0.046977,0.008961,0.001388,0.087845,0,0,"[{'a': 0, 'h': 0, 'p': 0.4532}, {'a': 0, 'h': ...","{'atk_away': -1.3299315104386522, 'atk_home': ...",None,2,1,3,H,True,False,False,False,False,2026-06-15 03:49:40.663774+00:00
3051,308,901638,2022-07-17,1512,13,2026-06-15 03:48:19.693527+00:00,0.486114,1.511607,0.112804,0.263913,0.623282,0.597737,0.322706,0.142465,0.052447,0.304437,0,1,"[{'a': 1, 'h': 0, 'p': 0.2007}, {'a': 2, 'h': ...","{'atk_away': 0.513112295717973, 'atk_home': -0...",None,1,1,2,D,True,False,True,False,False,2026-06-15 03:49:40.663774+00:00
3052,306,901636,2022-07-17,1493,1507,2026-06-15 03:48:19.692664+00:00,1.755013,2.826021,0.224187,0.176469,0.599343,0.945039,0.835304,0.671132,0.483112,0.780305,1,2,"[{'a': 2, 'h': 1, 'p': 0.0718}, {'a': 3, 'h': ...","{'atk_away': 0.6361894246832311, 'atk_home': -...",None,0,1,1,A,False,True,False,False,False,2026-06-15 03:49:40.663774+00:00
3053,250,865834,2022-06-13,20,30,2026-06-15 03:48:17.545752+00:00,1.767556,0.646383,0.636168,0.240940,0.122892,0.702624,0.433927,0.224194,0.097622,0.402818,1,0,"[{'a': 0, 'h': 1, 'p': 0.1501}, {'a': 0, 'h': ...","{'atk_away': 0.5931288650474474, 'atk_home': 1...",None,0,0,0,D,False,False,True,True,False,2026-06-15 03:49:40.663774+00:00


## 3. Matches by calendar year
Note how qualifier 'season' buckets carry matches into 2025–2026 — that's our recent form.

In [3]:
q('''
SELECT EXTRACT(YEAR FROM match_date)::int AS year,
       count(*) AS matches,
       count(*) FILTER (WHERE status IN ('FT','AET','PEN')) AS finished,
       round(avg(home_goals + away_goals) FILTER (WHERE status IN ('FT','AET','PEN'))::numeric, 2) AS avg_total_goals
FROM football.matches GROUP BY year ORDER BY year
''')

,year,matches,finished,avg_total_goals
0,2019,136,136,2.74
1,2020,21,21,3.43
2,2021,370,363,2.76
3,2022,931,861,2.55
4,2023,1036,997,2.78
5,2024,982,964,2.65
6,2025,451,441,3.02
7,2026,29,29,2.83


## 4. Matches by competition (importance weight)

In [4]:
q('''
SELECT competition, league_id, competition_weight AS weight, count(*) AS matches,
       min(match_date) AS first, max(match_date) AS last
FROM football.matches GROUP BY competition, league_id, competition_weight
ORDER BY competition_weight DESC, matches DESC
''')

,competition,league_id,weight,matches,first,last
0,World Cup,1,3.0,64,2022-11-20,2022-12-18
1,Africa Cup of Nations,6,2.5,52,2024-01-13,2024-02-11
2,Asian Cup,7,2.5,51,2024-01-12,2024-02-10
3,Euro Championship,4,2.5,51,2024-06-14,2024-07-14
4,Copa America,9,2.5,32,2024-06-21,2024-07-15
5,CONCACAF Gold Cup,22,2.5,31,2023-06-25,2023-07-16
6,UEFA Nations League,5,2.0,356,2022-06-01,2026-03-31
7,CONCACAF Nations League,536,2.0,320,2022-06-02,2025-03-24
8,World Cup - Qualification Africa,29,1.8,431,2019-09-04,2025-11-16
9,Euro Championship - Qualification,960,1.8,239,2023-03-23,2024-03-26


## 5. Sample: most recent finished matches

In [5]:
q('''
SELECT m.match_date, m.competition,
       th.name AS home, m.home_goals, m.away_goals, ta.name AS away, m.status
FROM football.matches m
JOIN football.teams th ON th.team_id = m.home_team_id
JOIN football.teams ta ON ta.team_id = m.away_team_id
WHERE m.status IN ('FT','AET','PEN')
ORDER BY m.match_date DESC LIMIT 15
''')

,match_date,competition,home,home_goals,away_goals,away,status
0,2026-06-04,Asian Cup - Qualification,Lebanon,0,2,Yemen,FT
1,2026-03-31,World Cup - Qualification Europe,Bosnia & Herzegovina,1,1,Italy,PEN
2,2026-03-31,World Cup - Qualification Europe,Kosovo,0,1,Türkiye,FT
3,2026-03-31,World Cup - Qualification Europe,Czechia,1,1,Denmark,PEN
4,2026-03-31,World Cup - Qualification Europe,Sweden,3,2,Poland,FT
5,2026-03-31,UEFA Nations League,Latvia,1,0,Gibraltar,FT
6,2026-03-31,UEFA Nations League,Luxembourg,3,0,Malta,FT
7,2026-03-31,Asian Cup - Qualification,Pakistan,1,2,Myanmar,FT
8,2026-03-31,Asian Cup - Qualification,Maldives,2,1,Timor-Leste,FT
9,2026-03-31,Asian Cup - Qualification,Chinese Taipei,1,3,Sri Lanka,FT


## 6. WC2026 qualifiers played in 2025–2026 (recent competitive form)

In [6]:
q('''
SELECT m.match_date, m.competition,
       th.name AS home, m.home_goals, m.away_goals, ta.name AS away
FROM football.matches m
JOIN football.teams th ON th.team_id = m.home_team_id
JOIN football.teams ta ON ta.team_id = m.away_team_id
WHERE m.competition LIKE 'World Cup - Qualification%' AND m.match_date >= '2025-01-01'
ORDER BY m.match_date DESC LIMIT 15
''')

,match_date,competition,home,home_goals,away_goals,away
0,2026-03-31,World Cup - Qualification Europe,Bosnia & Herzegovina,1,1,Italy
1,2026-03-31,World Cup - Qualification Europe,Kosovo,0,1,Türkiye
2,2026-03-31,World Cup - Qualification Europe,Czechia,1,1,Denmark
3,2026-03-31,World Cup - Qualification Europe,Sweden,3,2,Poland
4,2026-03-26,World Cup - Qualification Europe,Slovakia,3,4,Kosovo
5,2026-03-26,World Cup - Qualification Europe,Ukraine,1,3,Sweden
6,2026-03-26,World Cup - Qualification Europe,Czechia,2,2,Rep. Of Ireland
7,2026-03-26,World Cup - Qualification Europe,Italy,2,0,Northern Ireland
8,2026-03-26,World Cup - Qualification Europe,Wales,1,1,Bosnia & Herzegovina
9,2026-03-26,World Cup - Qualification Europe,Poland,2,1,Albania


## 7. Teams sample

In [7]:
q('''
SELECT team_id, name, fifa_code, country
FROM football.teams WHERE country IS NOT NULL ORDER BY name LIMIT 20
''')

,team_id,name,fifa_code,country
0,1553,Afghanistan,AFG,Afghanistan
1,778,Albania,ALB,Albania
2,10354,Albania U19,ALB,Albania
3,1532,Algeria,ALG,Algeria
4,11017,Algeria U20,NaN,Algeria
5,1110,Andorra,AND,Andorra
6,1529,Angola,ANG,Angola
7,26,Argentina,ARG,Argentina
8,10187,Argentina U23,NaN,Argentina
9,1094,Armenia,ARM,Armenia


## 8. Goal-scoring leaders (avg goals for / against per match, min 20 matches)

In [8]:
q('''
WITH per_team AS (
  SELECT home_team_id AS team_id, home_goals AS gf, away_goals AS ga FROM football.matches WHERE status IN ('FT','AET','PEN')
  UNION ALL
  SELECT away_team_id, away_goals, home_goals FROM football.matches WHERE status IN ('FT','AET','PEN')
)
SELECT t.name, count(*) AS matches,
       round(avg(p.gf)::numeric,2) AS gf_per_game,
       round(avg(p.ga)::numeric,2) AS ga_per_game,
       round((avg(p.gf)-avg(p.ga))::numeric,2) AS goal_diff
FROM per_team p JOIN football.teams t ON t.team_id = p.team_id
GROUP BY t.name HAVING count(*) >= 20
ORDER BY goal_diff DESC LIMIT 15
''')

,name,matches,gf_per_game,ga_per_game,goal_diff
0,Japan,42,2.83,0.74,2.10
1,Haiti,26,2.88,1.12,1.77
2,Argentina,44,2.20,0.43,1.77
3,Portugal,48,2.56,0.83,1.73
4,Spain,50,2.56,0.84,1.72
5,Iran,42,2.45,0.74,1.71
6,Norway,38,2.66,1.03,1.63
7,Brazil,37,2.22,0.68,1.54
8,Algeria,40,2.25,0.83,1.43
9,England,48,2.06,0.69,1.38


## 9. match_stats (corners/cards/possession)
Trickling in within the API daily cap — likely empty or partial for now.

In [9]:
print('statted matches:', q("SELECT count(DISTINCT fixture_id) AS n FROM football.match_stats").iloc[0]['n'])
q('''
SELECT s.fixture_id, t.name AS team, s.is_home, s.possession, s.shots_total,
       s.shots_on_target, s.corners, s.fouls, s.yellow_cards, s.red_cards, s.xg
FROM football.match_stats s JOIN football.teams t ON t.team_id = s.team_id
LIMIT 10
''')

statted matches: 0


,fixture_id,team,is_home,possession,shots_total,shots_on_target,corners,fouls,yellow_cards,red_cards,xg


## 10. team_ratings & predictions
Populated once `sandy.football.ratings.fit_and_persist()` and the predictor run (F3).

In [10]:
print('team_ratings rows:', q("SELECT count(*) AS n FROM football.team_ratings").iloc[0]['n'])
print('predictions rows:', q("SELECT count(*) AS n FROM football.match_predictions").iloc[0]['n'])

team_ratings rows: 0
predictions rows: 0
